# PortfolioPulse — raw data exploration

**This notebook is for showing working, not for running the pipeline.**

It profiles the raw file before any transformation: shape, dtypes, missing
values, and the distributions that decided how the schema was built.

No business logic lives here. Every reporting metric is computed in `sql/`.
To build the warehouse run `python src/ingest.py`, not this notebook.


In [ ]:
import pandas as pd

RAW = '../data/mortgage.csv'   # see DATA.md for how to obtain it
df = pd.read_csv(RAW)
df.shape


## What one row is

The first question to settle: is this one row per loan, or one row per loan
per period? Everything downstream depends on the answer.


In [ ]:
df.head()


In [ ]:
print('rows      :', len(df))
print('loans     :', df['id'].nunique())
print('rows/loan :', round(len(df) / df['id'].nunique(), 1))


More rows than loans, so this is a **panel** — one row per loan per period.
That is what makes roll rates and vintage curves possible at all.


## Dtypes


In [ ]:
df.dtypes


## Missing values

Checked before assuming any column is safe to band or join on.


In [ ]:
missing = df.isna().sum()
missing[missing > 0] if (missing > 0).any() else 'no missing values in any column'


## Is `time` monthly or quarterly?

The column is an integer with no stated unit. Loan term settles it: a
30-year mortgage should show 360 periods if monthly, 120 if quarterly.


In [ ]:
(df['mat_time'] - df['orig_time']).value_counts().head(8)


Dominant value is **120**, so the periods are **quarters**, not months.
The fact table is named `fact_loan_quarter` for this reason.


## Status distribution

`status_time` is the loan's state in that period. This is the equivalent of
checking delinquency status and zero balance codes on a servicing file —
note how few states exist here.


In [ ]:
df['status_time'].value_counts().rename({0:'0 active',1:'1 default',2:'2 payoff'})


In [ ]:
# Final state per loan, which is what a lifetime default rate is built from
last = df.sort_values('time').groupby('id')['status_time'].last()
(last.value_counts(normalize=True) * 100).round(1)


Only three states, and no days-past-due field anywhere in the file.
**No delinquency buckets exist**, so no 30+/60+/90+ arrears rate can be
reported and roll rates become status transitions. Recorded in DATA.md.


## Recovering the calendar

The periods are deidentified. But the file carries the US unemployment rate
and a house price index per period, and those can be anchored against known
history.


In [ ]:
macro = df.groupby('time')[['uer_time','hpi_time','gdp_time']].mean().round(2)
macro.loc[[1, 14, 25, 39, 60]]


In [ ]:
print('unemployment peaks at period', macro['uer_time'].idxmax(),
      '->', macro['uer_time'].max(), '(US 2009Q4: 9.9)')
print('house prices peak at period', macro['hpi_time'].idxmax(),
      '-> US housing peaked 2006Q2')


Unemployment peaks at 10.0 and house prices peak 14 periods earlier, which
puts **period 1 at 2000Q2**. Used only for labelling — see
`docs/Known Assumptions.md`.


## Origination cohorts

Which vintages exist, and do they differ enough to be worth contrasting?


In [ ]:
cohort = df.groupby('id').agg(orig_time=('orig_time','first'),
                              final=('status_time','last'))
by_vintage = cohort[cohort['orig_time'].between(1, 60)].groupby('orig_time').agg(
    loans=('final','size'), default_pct=('final', lambda s: round(100*(s==1).mean(), 1)))
by_vintage[by_vintage['loans'] >= 300]


Lifetime default rate rises steadily across origination quarters. The two
ends of that range become the contrast cohorts in `src/config.py`.


## Things that looked wrong

Profiling turned up two source defects. Neither is fixed here — handling is
decided in the pipeline and logged to `dq_results`. See
`docs/Data Quality Rules.md`.


In [ ]:
dupes = df.duplicated(subset=['id','time']).sum()
print('duplicate loan-period rows :', dupes)

spans = df.groupby('id')['time'].agg(['count','min','max'])
gaps = (spans['count'] != spans['max'] - spans['min'] + 1).sum()
print('loans with panel gaps      :', gaps)


---

**Next:** `python src/ingest.py` builds the star schema, `python src/views.py`
creates the reporting views. Nothing in this notebook is used by the pipeline.
